# Programmatic Tool Calling: Tool Search vs. Eager Tool Loading

**Scenario:** Inventory replenishment with a large function catalog

This experiment asks a focused question: when a model can choose among many function tools,
does hosted **Tool Search** let **Programmatic Tool Calling** load only the inventory namespace
before generating its JavaScript program?

The primary comparison is `programmatic_eager` versus `programmatic_tool_search`. A
`direct_eager` arm is available only as an optional reference. Live API execution is opt-in.


## 1. Experimental contract

- Use the same deterministic small inventory dataset in every arm.

- Scale the available function catalog to 20, 50, 100, or 200 functions, with at most five
  functions per namespace.

- Treat the three inventory lookup functions as required. The two other inventory functions
  are safe read-only helpers; loading them produces an efficiency warning, not a quality failure.

- Fail a run if Tool Search loads an unrelated namespace or if execution deviates from the exact
  required tool/SKU set.

- For the combined arm, expect this semantic order:
  `tool_search_call → tool_search_output → program → function_call → program_output → message`.

- Use an explicit prompt-cache breakpoint after the stable developer contract. Cold/write and
  warm/read runs share a key, while arm, catalog size, and repetition use isolated keys.


In [1]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

from ptc_benchmark.inventory_tool_search import (
    CATALOG_SIZES,
    REQUIRED_INVENTORY_TOOLS,
    build_inventory_tool_search_scenario,
)
from ptc_benchmark.inventory_tool_search_evaluation import evaluate_inventory_tool_search_run
from ptc_benchmark.inventory_tool_search_runner import (
    InventoryToolSearchRunner,
    ToolSearchRunConfig,
    comparison_order,
    semantic_timeline,
)
from ptc_benchmark.pricing import estimate_usage_cost, load_pricing_catalog
from ptc_benchmark.reporting import markdown_table

ROOT = Path.cwd()
if not (ROOT / "src" / "ptc_benchmark").exists():
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env.local")
PRICING = load_pricing_catalog(ROOT / "pricing" / "openai_pricing_2026-08-14.json")


In [2]:
# Enable only when you intentionally want API calls that may incur cost.
RUN_LIVE = True
# Enable after RUN_LIVE to execute cold/write and warm/read repetitions.
RUN_REPEATED_COMPARISON = True
# Enable only when you intentionally want the full 20/50/100/200 catalog sweep.
RUN_ALL_SCALES = False
# Optional reference arm; the primary comparison remains the two Programmatic arms.
INCLUDE_DIRECT_BASELINE = False

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6")
CATALOG_SIZE = 100
REPETITIONS = 3
DATASET_SCALE = "medium" # small=3 SKUs, medium=10, large=30
MAX_REQUESTS = 16

assert CATALOG_SIZE in CATALOG_SIZES
print({
    "RUN_LIVE": RUN_LIVE,
    "RUN_REPEATED_COMPARISON": RUN_REPEATED_COMPARISON,
    "RUN_ALL_SCALES": RUN_ALL_SCALES,
    "INCLUDE_DIRECT_BASELINE": INCLUDE_DIRECT_BASELINE,
    "MODEL": MODEL,
    "CATALOG_SIZE": CATALOG_SIZE,
    "REPETITIONS": REPETITIONS,
})


{'RUN_LIVE': True, 'RUN_REPEATED_COMPARISON': True, 'RUN_ALL_SCALES': False, 'INCLUDE_DIRECT_BASELINE': False, 'MODEL': 'gpt-5.6', 'CATALOG_SIZE': 100, 'REPETITIONS': 3}


## 2. Inspect the deterministic catalog

The inventory namespace always contains the three required functions and two same-domain helpers.
Every remaining namespace is a deterministic, read-only distractor. The code below inspects the
tool surface without making an API request.


In [3]:
catalog_rows = []
for size in CATALOG_SIZES:
    scenario = build_inventory_tool_search_scenario(
        catalog_size=size,
        inventory_scale=DATASET_SCALE,
    )
    combined_tools = scenario.tool_definitions("programmatic_tool_search")
    namespaces = [tool for tool in combined_tools if tool["type"] == "namespace"]
    functions = [function for namespace in namespaces for function in namespace["tools"]]
    catalog_rows.append({
        "catalog_size": size,
        "namespaces": len(namespaces),
        "functions": len(functions),
        "deferred_functions": sum(fn.get("defer_loading") is True for fn in functions),
        "hosted_tools": ", ".join(tool["type"] for tool in combined_tools if tool["type"] != "namespace"),
    })

display(Markdown(markdown_table(catalog_rows)))


| catalog_size | namespaces | functions | deferred_functions | hosted_tools |
| --- | --- | --- | --- | --- |
| 20 | 4 | 20 | 20 | tool_search, programmatic_tool_calling |
| 50 | 10 | 50 | 50 | tool_search, programmatic_tool_calling |
| 100 | 20 | 100 | 100 | tool_search, programmatic_tool_calling |
| 200 | 40 | 200 | 200 | tool_search, programmatic_tool_calling |

## 3. Compare the tool surfaces

`programmatic_eager` exposes all detailed schemas to the model immediately. In
`programmatic_tool_search`, every function has `defer_loading: true`; the model initially sees
namespace summaries plus the hosted Tool Search and PTC tools. The combined arm deliberately does
not force a first tool choice: the developer contract tells the model to search first, and the
quality gate verifies what actually happened.


In [4]:
scenario = build_inventory_tool_search_scenario(
    catalog_size=CATALOG_SIZE,
    inventory_scale=DATASET_SCALE,
)

surface_rows = []
for arm in ("programmatic_eager", "programmatic_tool_search", "direct_eager"):
    tools = scenario.tool_definitions(arm)
    functions = [fn for ns in tools if ns["type"] == "namespace" for fn in ns["tools"]]
    surface_rows.append({
        "arm": arm,
        "function_schemas": len(functions),
        "deferred": sum(fn.get("defer_loading") is True for fn in functions),
        "tool_search": any(tool["type"] == "tool_search" for tool in tools),
        "programmatic": any(tool["type"] == "programmatic_tool_calling" for tool in tools),
        "allowed_caller": functions[0]["allowed_callers"][0],
    })

display(Markdown(markdown_table(surface_rows)))


| arm | function_schemas | deferred | tool_search | programmatic | allowed_caller |
| --- | --- | --- | --- | --- | --- |
| programmatic_eager | 100 | 0 | False | True | programmatic |
| programmatic_tool_search | 100 | 100 | True | True | programmatic |
| direct_eager | 100 | 0 | False | False | direct |

## 4. Live single comparison

This cell runs the two primary arms once and optionally includes the Direct baseline. It reports
quality before tokens, estimated cost, or latency. The pricing snapshot is dated and the dollar
values are estimates. With `RUN_LIVE = False`, the saved notebook remains safe and reproducible.


In [5]:
live_runs = {}
live_evaluations = {}

if RUN_LIVE:
    from openai import OpenAI

    runner = InventoryToolSearchRunner(OpenAI())
    arms = ["programmatic_eager", "programmatic_tool_search"]
    if INCLUDE_DIRECT_BASELINE:
        arms.append("direct_eager")

    rows = []
    for arm in arms:
        run = runner.run(
            arm=arm,
            scenario=scenario,
            config=ToolSearchRunConfig(model=MODEL, max_requests=MAX_REQUESTS),
            experiment_id="single-comparison",
            repetition=1,
        )
        evaluation = evaluate_inventory_tool_search_run(run, scenario)
        cost = estimate_usage_cost(run.usage, MODEL, PRICING)
        live_runs[arm] = run
        live_evaluations[arm] = evaluation
        rows.append({
            "arm": arm,
            "quality_passed": evaluation.passed,
            "loaded_tools": len(run.loaded_tools),
            "executed_tools": len(run.tool_calls),
            "input_tokens": run.usage.input_tokens,
            "cached_tokens": run.usage.cached_input_tokens,
            "cache_write_tokens": run.usage.cache_write_input_tokens,
            "output_tokens": run.usage.output_tokens,
            "reasoning_tokens": run.usage.reasoning_output_tokens,
            "estimated_cost_usd": round(cost.total_cost, 6),
            "end_to_end_seconds": round(run.total_latency_seconds, 3),
        })
    display(Markdown(markdown_table(rows)))
else:
    print("Live comparison not run. Set RUN_LIVE = True to enable API calls.")


| arm | quality_passed | loaded_tools | executed_tools | input_tokens | cached_tokens | cache_write_tokens | output_tokens | reasoning_tokens | estimated_cost_usd | end_to_end_seconds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| programmatic_eager | True | 0 | 30 | 20187 | 9807 | 9676 | 629 | 0 | 0.087768 | 18.562 |
| programmatic_tool_search | True | 5 | 30 | 6401 | 0 | 4782 | 725 | 58 | 0.059733 | 15.697 |

## 5. Inspect the semantic timeline

For the combined arm, inspect the normalized event sequence rather than assuming that deferred
tools were loaded selectively. A passing trace shows hosted Tool Search at the top level, then the
generated program, program-owned function calls, the program result, and finally the assistant
message. Tool Search cannot be invoked from inside an already-running JavaScript program.


In [6]:
combined_run = live_runs.get("programmatic_tool_search")
if combined_run is not None:
    display(Markdown(markdown_table(semantic_timeline(combined_run))))
    evaluation = live_evaluations["programmatic_tool_search"]
    print("warnings:", evaluation.warnings or "none")
    print("failures:", evaluation.failures or "none")
else:
    print("Semantic timeline not available because the live comparison was not run.")


| sequence | request | type | detail |
| --- | --- | --- | --- |
| 1 | 1 | reasoning |  |
| 2 | 1 | tool_search_call | {"paths": ["inventory"]} |
| 3 | 1 | tool_search_output | inventory.get_inventory, inventory.get_weekly_demand, inventory.get_inbound_shipments, inventory.get_safety_stock_policy, inventory.get_inventory_dataset_metadata |
| 4 | 1 | reasoning |  |
| 5 | 1 | program | generated JavaScript |
| 6 | 1 | function_call | inventory.get_inventory |
| 7 | 1 | function_call | inventory.get_weekly_demand |
| 8 | 1 | function_call | inventory.get_inventory |
| 9 | 1 | function_call | inventory.get_inbound_shipments |
| 10 | 1 | function_call | inventory.get_inbound_shipments |
| 11 | 1 | function_call | inventory.get_weekly_demand |
| 12 | 1 | function_call | inventory.get_weekly_demand |
| 13 | 1 | function_call | inventory.get_weekly_demand |
| 14 | 1 | function_call | inventory.get_inbound_shipments |
| 15 | 1 | function_call | inventory.get_inventory |
| 16 | 1 | function_call | inventory.get_inventory |
| 17 | 1 | function_call | inventory.get_inbound_shipments |
| 18 | 1 | function_call | inventory.get_inventory |
| 19 | 1 | function_call | inventory.get_inventory |
| 20 | 1 | function_call | inventory.get_inbound_shipments |
| 21 | 1 | function_call | inventory.get_weekly_demand |
| 22 | 1 | function_call | inventory.get_weekly_demand |
| 23 | 1 | function_call | inventory.get_inbound_shipments |
| 24 | 1 | function_call | inventory.get_inbound_shipments |
| 25 | 1 | function_call | inventory.get_weekly_demand |
| 26 | 1 | function_call | inventory.get_inventory |
| 27 | 1 | function_call | inventory.get_inbound_shipments |
| 28 | 1 | function_call | inventory.get_inventory |
| 29 | 1 | function_call | inventory.get_weekly_demand |
| 30 | 1 | function_call | inventory.get_weekly_demand |
| 31 | 1 | function_call | inventory.get_inbound_shipments |
| 32 | 1 | function_call | inventory.get_inbound_shipments |
| 33 | 1 | function_call | inventory.get_inventory |
| 34 | 1 | function_call | inventory.get_weekly_demand |
| 35 | 1 | function_call | inventory.get_inventory |
| 36 | 2 | program_output | structured program result |
| 37 | 2 | message | final assistant message |

warnings: ('Tool Search loaded extra inventory tools (efficiency warning only): inventory.get_inventory_dataset_metadata, inventory.get_safety_stock_policy',)
failures: none


## 6. Repeated cold/write and warm/read protocol

Each repetition alternates arm order to reduce order bias. For a given arm, catalog size, and
repetition, the cold/write and warm/read calls reuse the same explicit cache key. Different arms,
sizes, and repetitions never share a key. `RUN_ALL_SCALES` expands the protocol from the selected
catalog size to 20/50/100/200.

The first run may write the stable prefix (`cache_write_tokens`); the second can read it
(`cached_tokens`). Interpret Tool Search latency separately from context and cost effects.

`estimated_cost_usd` separates input tokens into uncached, cache-read, and cache-write classes,
then applies the model-specific rates from the pricing snapshot and adds output cost:

`((input_tokens - cached_tokens - cache_write_tokens) × input_rate + cached_tokens × cached_input_rate + cache_write_tokens × cache_write_rate + output_tokens × output_rate) / 1,000,000`. For cache-written tokens, the cost is calculated using a price of 1.25× the ordinary
input-token price. This 1.25× price is the total charge for those tokens, not an additional
surcharge added on top of the ordinary input-token cost.


### Explicit cache breakpoint placement

[`InventoryToolSearchRunner`](../api/ptc_benchmark/inventory_tool_search_runner.html#L84) attaches the breakpoint to the developer `input_text`. The complete
developer instructions are therefore the stable portion before the logical boundary, and the user
prompt follows it as the dynamic suffix:

```
input_items = [
    {
        "role": "developer",
        "content": [{
            "type": "input_text",
            "text": instructions,
            "prompt_cache_breakpoint": {"mode": "explicit"},
        }],
    },
    {"role": "user", "content": [{"type": "input_text", "text": user_input}]},
]
```

For `programmatic_tool_search`, `instructions` already contains the task contract, the
Programmatic Tool Calling orchestration, and the `tool_search_contract`. The resulting logical
boundary is:

```
task contract + Programmatic orchestration + Tool Search contract
----------------------------------------------------------------- explicit breakpoint
user prompt
```

The request also enables explicit caching and supplies the stable experiment key. Both the request
option and the content breakpoint are required by this implementation:

```
request = {
    ...
    "input": input_items,
    "tools": tools,
    "prompt_cache_key": cache_key,
    "prompt_cache_options": {"mode": "explicit"},
}
```

For the combined arm, every function schema is marked `defer_loading: true`, then hosted Tool
Search and Programmatic Tool Calling are added as top-level tools:

```
if arm == "programmatic_tool_search":
    function["defer_loading"] = True

definitions.extend([
    {"type": "tool_search"},
    {"type": "programmatic_tool_calling"},
])
```

This deferred-loading configuration is related to Tool Search but does not itself define the cache
breakpoint. The cache key is generated as
`ptc-ts:<experiment_id>:<catalog_size>:<arm>:r<repetition>`. A cold/write and its paired warm/read
run reuse that key, while different arms, catalog sizes, and repetitions remain isolated. Runner
tests verify both `prompt_cache_options` and the breakpoint attached to the first developer content
item.

```
stable developer contract
  └─ explicit breakpoint
dynamic user prompt
  └─ hosted Tool Search
      └─ selected tool definitions
          └─ Programmatic Tool Calling
```


In [7]:
protocol_rows = []
sizes = CATALOG_SIZES if RUN_ALL_SCALES else (CATALOG_SIZE,)
for size in sizes:
    for repetition in range(1, REPETITIONS + 1):
        protocol_rows.append({
            "catalog_size": size,
            "repetition": repetition,
            "arm_order": " → ".join(comparison_order(repetition)),
            "cache_phases": "cold/write → warm/read",
        })
display(Markdown(markdown_table(protocol_rows)))


| catalog_size | repetition | arm_order | cache_phases |
| --- | --- | --- | --- |
| 100 | 1 | programmatic_eager → programmatic_tool_search | cold/write → warm/read |
| 100 | 2 | programmatic_tool_search → programmatic_eager | cold/write → warm/read |
| 100 | 3 | programmatic_eager → programmatic_tool_search | cold/write → warm/read |

In [8]:
repeated_rows = []
if RUN_LIVE and RUN_REPEATED_COMPARISON:
    from openai import OpenAI

    runner = InventoryToolSearchRunner(OpenAI())
    for size in sizes:
        repeated_scenario = build_inventory_tool_search_scenario(
            catalog_size=size,
            inventory_scale=DATASET_SCALE,
        )
        for repetition in range(1, REPETITIONS + 1):
            for arm in comparison_order(repetition):
                for cache_phase in ("cold_write", "warm_read"):
                    run = runner.run(
                        arm=arm,
                        scenario=repeated_scenario,
                        config=ToolSearchRunConfig(model=MODEL, max_requests=MAX_REQUESTS),
                        experiment_id="repeated-comparison",
                        repetition=repetition,
                    )
                    evaluation = evaluate_inventory_tool_search_run(run, repeated_scenario)
                    cost = estimate_usage_cost(run.usage, MODEL, PRICING)
                    repeated_rows.append({
                        "catalog_size": size,
                        "repetition": repetition,
                        "cache_phase": cache_phase,
                        "arm": arm,
                        "quality_passed": evaluation.passed,
                        "loaded_tools": len(run.loaded_tools),
                        "input_tokens": run.usage.input_tokens,
                        "cached_tokens": run.usage.cached_input_tokens,
                        "cache_write_tokens": run.usage.cache_write_input_tokens,
                        "estimated_cost_usd": round(cost.total_cost, 6),
                        "end_to_end_seconds": round(run.total_latency_seconds, 3),
                    })
    arm_order = dict.fromkeys(row["arm"] for row in repeated_rows)
    for arm in arm_order:
        arm_rows = [
            {key: value for key, value in row.items() if key != "arm"}
            for row in repeated_rows
            if row["arm"] == arm
        ]
        display(Markdown(f"### `{arm}`\n\n{markdown_table(arm_rows)}"))
else:
    print("Repeated comparison not run. Enable RUN_LIVE and RUN_REPEATED_COMPARISON intentionally.")


### `programmatic_eager`

| catalog_size | repetition | cache_phase | quality_passed | loaded_tools | input_tokens | cached_tokens | cache_write_tokens | estimated_cost_usd | end_to_end_seconds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 100 | 1 | cold_write | True | 0 | 20179 | 9807 | 9676 | 0.087488 | 14.502 |
| 100 | 1 | warm_read | True | 0 | 20179 | 19483 | 0 | 0.031941 | 17.699 |
| 100 | 2 | cold_write | True | 0 | 20183 | 9807 | 9676 | 0.088018 | 13.424 |
| 100 | 2 | warm_read | True | 0 | 20168 | 19483 | 0 | 0.031467 | 14.372 |
| 100 | 3 | cold_write | True | 0 | 20183 | 9807 | 9676 | 0.088018 | 16.519 |
| 100 | 3 | warm_read | True | 0 | 20179 | 19483 | 0 | 0.031971 | 17.177 |

### `programmatic_tool_search`

| catalog_size | repetition | cache_phase | quality_passed | loaded_tools | input_tokens | cached_tokens | cache_write_tokens | estimated_cost_usd | end_to_end_seconds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 100 | 1 | cold_write | True | 5 | 6386 | 0 | 4782 | 0.059297 | 16.668 |
| 100 | 1 | warm_read | True | 5 | 6368 | 4782 | 0 | 0.031111 | 14.906 |
| 100 | 2 | cold_write | True | 5 | 6365 | 0 | 4782 | 0.058563 | 47.465 |
| 100 | 2 | warm_read | True | 5 | 6378 | 4782 | 0 | 0.031491 | 13.58 |
| 100 | 3 | cold_write | True | 5 | 6353 | 0 | 4782 | 0.058112 | 14.363 |
| 100 | 3 | warm_read | True | 5 | 6322 | 4782 | 0 | 0.029651 | 14.788 |

## 7. Interpretation and output convention

Use only quality-passing runs for cost comparisons. Compare loaded tool names, not merely the
presence of `tool_search_output`. A same-namespace helper warning indicates avoidable context, but
an unrelated namespace is a failure. Lower input tokens do not automatically mean lower total
cost because Tool Search and program generation can add output/reasoning tokens and latency.

When live evidence is intentionally exported, use the dedicated directory
`outputs/04_inventory_programmatic_tool_search/` and this filename shape:

`inventory_<catalog-size>_<cache-phase>_<section>_<execution-date>.md`

No placeholder or fabricated live result is saved by this notebook.


## 8. Saved small and medium analysis

The saved 100-function runs show that adding Tool Search to Programmatic Tool Calling reduced
input tokens by approximately 69% for both datasets while preserving quality in every run. It
reduced total estimated cost across cold/write and warm/read phases by 28.2% for small and 25.9%
for medium, with the largest savings occurring during cold cache writes.

Latency did not move in one direction: Tool Search was 9.3% slower overall for small but 11.7%
faster for medium. The most plausible interpretation is that the hosted search step adds a mostly
fixed cost that is visible in the smaller workload, while the larger workload can amortize that
cost and benefit more from reducing the active tool context from 100 definitions to five. This is
directional evidence rather than a causal conclusion because each cache phase has only three runs,
and the medium warm/read mean is sensitive to one slow eager run.

See the complete small and medium experiment analysis
for the aggregate table, cost and caching interpretation, latency caveats, and recommended use cases.
